<a href="https://colab.research.google.com/github/sulucay01/DI725-assignment1/blob/dev/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing

This notebook prepares the raw customer support conversations for downstream modeling.

The preprocessing pipeline includes:
- loading raw train and test datasets
- removing malformed conversations
- cleaning and normalizing text
- standardizing categorical variables
- creating simple text-based features
- encoding labels for the training set
- creating a stratified validation split
- saving processed datasets

In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

os.makedirs("data/processed", exist_ok=True)

## 1. Load Raw Data

In this section, we load the raw train and test datasets and inspect their basic structure.

In [2]:
train = pd.read_csv(
    "https://raw.githubusercontent.com/sulucay01/DI725-assignment1/main/data/raw/train.csv"
)
test = pd.read_csv(
    "https://raw.githubusercontent.com/sulucay01/DI725-assignment1/main/data/raw/test.csv"
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain columns:")
print(train.columns.tolist())

train.head(3)

Train shape: (970, 11)
Test shape: (30, 11)

Train columns:
['issue_area', 'issue_category', 'issue_sub_category', 'issue_category_sub_category', 'customer_sentiment', 'product_category', 'product_sub_category', 'issue_complexity', 'agent_experience_level', 'agent_experience_level_desc', 'conversation']


,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or email address during login,Mobile Number and Email Verification -> Verification requirement for mobile number or email address during login,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, possess solid troubleshooting skills, and seek guidance from more experienced team members when needed.","Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today?\n\nCustomer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG),..."
1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked to ship the item,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, possess solid troubleshooting skills, and seek guidance from more experienced team members when needed.",Agent: Thank you for calling BrownBox customer support. My name is Alex. How may I assist you today?\n\nCustomer: Hi Alex. I recently received an email from BrownBox requesting me to ship back the...
2,Cancellations and returns,Replacement and Return Process,Inability to click the 'Cancel' button,Replacement and Return Process -> Inability to click the 'Cancel' button,neutral,Appliances,Juicer/Mixer/Grinder,medium,experienced,"confidently handles complex customer issues, excel in de-escalation, and possess the ability to empathize with customers, providing them with effective solutions and support.","Agent: Thank you for calling BrownBox Customer Support. My name is Sarah. How may I assist you today?\n\nCustomer: Hi Sarah, I am calling because I am unable to click the 'Cancel' button for my Ju..."


## 2. Define Preprocessing Functions

To make the preprocessing pipeline reusable and easier to maintain, we define helper functions for:
- removing malformed rows
- cleaning conversation text
- normalizing categorical variables
- generating simple text-derived features
- applying the full preprocessing pipeline

In [3]:
def remove_invalid_conversations(df):
    df = df.copy()

    mask_no_customer = (
        df["conversation"].str.contains(r"\bAgent\s*:", regex=True, na=False, case=False)
        & ~df["conversation"].str.contains(r"\bCustomer\s*:", regex=True, na=False, case=False)
    )

    removed_count = mask_no_customer.sum()
    df = df.loc[~mask_no_customer].reset_index(drop=True)

    return df, removed_count

In [4]:
def clean_conversation(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Convert escaped line breaks first
    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")

    # Then normalize real line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace speaker tags
    text = re.sub(r"\bAgent\s*:", "[AGENT]", text, flags=re.IGNORECASE)
    text = re.sub(r"\bCustomer\s*:", "[CUSTOMER]", text, flags=re.IGNORECASE)

    # Mask patterns
    text = re.sub(
        r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
        "[EMAIL]",
        text
    )
    text = re.sub(
        r"(?<!\w)(?:\+\d{1,3}\s*)?(?:\(\d{3}\)|\d{3})[-.\s]\d{3}[-.\s]\d{4}(?!\w)",
        "[PHONE]",
        text
    )
    text = re.sub(r"\b\d{5,}\b", "[NUMBER]", text)
    text = re.sub(
        r"\b[A-Z]{2,}[A-Z0-9-]*\d+[A-Z0-9-]*\b|\b[A-Z]{2,}-\d+[A-Z0-9-]*\b",
        "[ID]",
        text
    )

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()

In [5]:
def normalize_categorical_columns(df):
    """
    Standardize categorical columns by filling missing values
    and trimming extra whitespace.
    """
    df = df.copy()

    for col in ["issue_area", "issue_category"]:
        if col in df.columns:
            df[col] = df[col].fillna("unknown").astype(str).str.strip()

    return df

In [6]:
def add_text_features(df):
    """
    Create simple text-derived features from the cleaned conversation text.
    """
    df = df.copy()

    df["char_len"] = df["clean_text"].astype(str).apply(len)
    df["word_len"] = df["clean_text"].astype(str).apply(lambda x: len(x.split()))
    df["line_count"] = df["clean_text"].astype(str).apply(lambda x: x.count("\n") + 1)

    return df

In [7]:
def preprocess_dataframe(df, dataset_name="dataset"):
    """
    Apply the full preprocessing pipeline to a dataframe.
    """
    df = df.copy()

    print(f"\nPreprocessing {dataset_name}...")
    print(f"Initial shape: {df.shape}")

    # Remove malformed rows
    df, removed_invalid = remove_invalid_conversations(df)
    print(f"Removed malformed rows: {removed_invalid}")
    print(f"Shape after malformed-row removal: {df.shape}")

    # Remove duplicate raw conversations
    before_raw_dedup = len(df)
    df = df.drop_duplicates(subset=["conversation"]).reset_index(drop=True)
    print(f"Removed duplicate raw conversations: {before_raw_dedup - len(df)}")
    print(f"Shape after raw deduplication: {df.shape}")

    # Normalize categorical columns
    df = normalize_categorical_columns(df)

    # Clean text
    df["clean_text"] = df["conversation"].apply(clean_conversation)

    # Remove duplicate cleaned conversations
    before_clean_dedup = len(df)
    df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)
    print(f"Removed duplicate cleaned conversations: {before_clean_dedup - len(df)}")
    print(f"Shape after cleaned-text deduplication: {df.shape}")

    # Add lightweight text features
    df = add_text_features(df)

    return df

## 3. Preprocess Training and Test Data

We now apply the same preprocessing pipeline to both the training and test sets.

The pipeline:
- removes malformed rows
- removes duplicates
- normalizes categorical fields
- cleans conversation text
- adds simple text-based features

## 4. Encode Target Labels

The target variable `customer_sentiment` is converted into numeric labels for model training.

We also save the label mapping so that the relationship between class names and encoded values remains explicit and reproducible.

## 5. Create Train/Validation Split

To evaluate models fairly, the processed training data is split into train and validation sets using stratification on the target label.
This helps preserve the class distribution in both subsets.

## 6. Save Processed Datasets

Finally, we save the processed train, validation, and test datasets, along with the label mapping, for downstream modeling experiments.

## 7. Sanity Checks

As a final check, we inspect a few cleaned examples and verify the final column structure of the processed training set.